In [1]:
!pip install optuna

In [45]:
from IPython.display import display

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

import optuna

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCH = 20

df = pd.read_csv('./dataset/data_cleaning.csv')
X, y = df.drop(columns='cardio', inplace=False), df['cardio']

display(X.head())

,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,ag_stage,bmi,age_years,age_groups
0,2,168,62.0,110,80,1,1,0,0,1,-1,22.0,50,2
1,1,156,85.0,140,90,3,1,0,0,1,1,35.0,55,2
2,1,165,64.0,130,70,3,1,0,0,0,-1,24.0,51,2
3,2,169,82.0,150,100,1,1,0,0,1,-1,29.0,48,1
4,1,156,56.0,100,60,1,1,0,0,0,-1,23.0,47,1


In [3]:
scaler = StandardScaler()
scaled_X = scaler.fit_transform(X)

scaled_X

array([[ 1.36403989,  0.45217522, -0.8537876 , ..., -1.04528311,
        -0.4201051 ,  0.66376649],
       [-0.73311639, -1.05776727,  0.75816977, ...,  1.43570517,
         0.31880809,  0.66376649],
       [-0.73311639,  0.0746896 , -0.71361739, ..., -0.66359261,
        -0.27232246,  0.66376649],
       ...,
       [ 1.36403989,  2.33960333,  2.15987184, ...,  0.67232416,
        -0.12453983,  0.66376649],
       [-0.73311639, -0.17696748, -0.15293657, ..., -0.09105685,
         1.20550393,  0.66376649],
       [-0.73311639,  0.7038323 , -0.15293657, ..., -0.47274735,
         0.46659073,  0.66376649]], shape=(69527, 14))

**Критерий Кайзера** (правило «собственное значение > 1») — это статистический метод в факторном анализе и методе главных компонент (PCA), используемый для определения оптимального количества сохраняемых факторов. Согласно критерию, следует оставлять только те факторы (компоненты), чьи собственные значения (eigenvalues) больше 1.

In [4]:
pca = PCA()
X_pca = pca.fit(scaled_X)

variance = X_pca.explained_variance_
cnt_component = variance[variance > 1].shape[0]

display(variance)
display(cnt_component)

array([2.91060763, 2.02914845, 1.61717642, 1.45517394, 1.28506544,
       1.08098511, 0.98791338, 0.65229114, 0.53982489, 0.52771527,
       0.47638302, 0.25450954, 0.17677353, 0.0066336 ])

6

In [5]:
pca = PCA(n_components=cnt_component)
X_pca = pca.fit_transform(scaled_X)

X_pca.shape, scaled_X.shape

((69527, 6), (69527, 14))

In [6]:
tensor_X = torch.from_numpy(X_pca).to(DEVICE)
tensor_y = torch.from_numpy(y.values).to(DEVICE)


print(DEVICE)

cpu


In [7]:
class MyData(Dataset):
    def __init__(self):
        self.x = tensor_X
        self.y = tensor_y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


data = MyData()
train_data, val_data, test_data = random_split(data, [.6, .2, .2])

train_loader = DataLoader(train_data, shuffle=True, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_data, shuffle=False, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, shuffle=False, batch_size=BATCH_SIZE)

In [8]:
def fit(model, optimizer, loss_func):
    model.train()
    for epoch in range(EPOCH):
        for data in train_loader:
            inputs, labels = data
            
            predict = model(inputs.float()).squeeze()
            loss = loss_func(predict, labels.float())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
    model.eval()



In [29]:
def objective(trial):
    model = nn.Sequential()
    
    n1 =  trial.suggest_int('n_neurons0', 50, 200)
    
    model.add_module('input_layer', nn.Linear(6, n1))
    model.add_module('act_input', nn.ReLU())
    
    n_layers = trial.suggest_int('n_layers', 1, 11)

    for i in range(n_layers):
        n2 =  trial.suggest_int(f'n_neurons{i+1}', 50, 200)
        
        model.add_module(f'hidden_layer{i+1}', nn.Linear(n1, n2))
        n1 = n2

        overfitting = trial.suggest_categorical(f'overfitting{i}', ['batchnorm', 'dropout', 'none'])
        if overfitting == 'batchnorm':
            model.add_module(f'batchnorm{i}', nn.BatchNorm1d(n2))
        elif overfitting == 'dropout':
            dropout_rate = trial.suggest_float(f'dropout_rate{i}', 0.1, 0.5)
            model.add_module(f'dropout{i}', nn.Dropout(dropout_rate))

        act_hidden = trial.suggest_categorical(f'act_hidden{i}', ['tanh', 'sigmoid', 'relu'])
        if act_hidden == 'tanh':
            model.add_module(f'act_hidden{i}', nn.Tanh())
        elif act_hidden == 'sigmoid':
            model.add_module(f'act_hidden{i}', nn.Sigmoid())
        else:
            model.add_module(f'act_hidden{i}', nn.ReLU())

    model.add_module('output_layer', nn.Linear(n1, 1))
    model.add_module('sigmoid', nn.Sigmoid())

    optimizer = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rms'])
    learning_rate = trial.suggest_float('lr', 1e-5, 1e-1)
    if optimizer == 'adam':
        optimization_func = optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer == 'sgd':
        optimization_func = optim.SGD(model.parameters(), lr=learning_rate)
    else:
        optimization_func = optim.RMSprop(model.parameters(), lr=learning_rate)

    loss_func = nn.BCELoss()
    fit(model, optimization_func, loss_func)

    loss = 0
    for i, data in enumerate(val_loader, 1):
        inputs, labels = data
        predict = model(inputs.float()).squeeze()
        loss += loss_func(predict, labels.float())
        
    return loss / i

    
        
study = optuna.create_study(study_name='my_study', direction='minimize')
study.optimize(objective, n_trials=200)

[I 2026-03-15 14:37:21,372] A new study created in memory with name: my_study
[I 2026-03-15 14:38:47,155] Trial 0 finished with value: 0.6939035654067993 and parameters: {'n_neurons0': 174, 'n_layers': 3, 'n_neurons1': 182, 'overfitting0': 'none', 'act_hidden0': 'relu', 'n_neurons2': 51, 'overfitting1': 'none', 'act_hidden1': 'sigmoid', 'n_neurons3': 62, 'overfitting2': 'none', 'act_hidden2': 'sigmoid', 'optimizer': 'adam', 'lr': 0.06730900168646328}. Best is trial 0 with value: 0.6939035654067993.
[I 2026-03-15 14:40:52,490] Trial 1 finished with value: 50.19354248046875 and parameters: {'n_neurons0': 114, 'n_layers': 9, 'n_neurons1': 89, 'overfitting0': 'batchnorm', 'act_hidden0': 'relu', 'n_neurons2': 92, 'overfitting1': 'dropout', 'dropout_rate1': 0.19068563496000035, 'act_hidden1': 'tanh', 'n_neurons3': 191, 'overfitting2': 'batchnorm', 'act_hidden2': 'relu', 'n_neurons4': 51, 'overfitting3': 'none', 'act_hidden3': 'relu', 'n_neurons5': 60, 'overfitting4': 'none', 'act_hidden4': '

In [30]:
hyperparam = study.best_params
hyperparam

{'n_neurons0': 187,
 'n_layers': 4,
 'n_neurons1': 65,
 'overfitting0': 'batchnorm',
 'act_hidden0': 'sigmoid',
 'n_neurons2': 166,
 'overfitting1': 'batchnorm',
 'act_hidden1': 'relu',
 'n_neurons3': 113,
 'overfitting2': 'dropout',
 'dropout_rate2': 0.1084533412002961,
 'act_hidden2': 'relu',
 'n_neurons4': 138,
 'overfitting3': 'none',
 'act_hidden3': 'relu',
 'optimizer': 'sgd',
 'lr': 0.011209685810525418}

In [46]:
model = nn.Sequential()

model.add_module('input_layer', nn.Linear(6, hyperparam['n_neurons0']))
model.add_module('act1', nn.ReLU())

prev_size = hyperparam['n_neurons0']
for i in range(1, hyperparam['n_layers']+1):
    current_size = hyperparam[f'n_neurons{i}']
    model.add_module(f'hidden_layer{i}', nn.Linear(prev_size, current_size))
    
    if f'overfitting{i-1}' in hyperparam:  
        if hyperparam[f'overfitting{i-1}'] == 'batchnorm':
            model.add_module(f'batchnorm{i}', nn.BatchNorm1d(current_size))
        elif hyperparam[f'overfitting{i-1}'] == 'dropout':
            model.add_module(f'dropout{i}', nn.Dropout(hyperparam[f'dropout_rate{i-1}']))
    
    
    if f'act_hidden{i-1}' in hyperparam: 
        if hyperparam[f'act_hidden{i-1}'] == 'sigmoid':
            model.add_module(f'act_hidden{i}', nn.Sigmoid())
        elif hyperparam[f'act_hidden{i-1}'] == 'relu':
            model.add_module(f'act_hidden{i}', nn.ReLU())
        else:
            model.add_module(f'act_hidden{i}', nn.Tanh())
    
    prev_size = current_size

model.add_module('output_layer', nn.Linear(prev_size, 1))
model.add_module('sigmoid', nn.Sigmoid())


model = model.to(DEVICE)

if hyperparam['optimizer'] == 'adam':
    optimization_func = optim.Adam(model.parameters(), lr=hyperparam['lr'])
elif hyperparam['optimizer'] == 'sgd':
    optimization_func = optim.SGD(model.parameters(), lr=hyperparam['lr'])
else:
    optimization_func = optim.RMSprop(model.parameters(), lr=hyperparam['lr'])

loss_func = nn.BCELoss()
fit(model, optimization_func, loss_func)

In [47]:
TP, TN, FP, FN = 0, 0, 0, 0

for x, y in test_loader:
    predict = model(x.float()).squeeze()
    y_pred = torch.where(predict > 0.5, 1, 0)
    for i in range(len(y)):
        if y[i] == 1 and y_pred[i] == 1:
            TP += 1
        elif y[i] == 1 and y_pred[i] == 0:
            FN += 1
        elif y[i] == 0 and y_pred[i] == 0:
            TN += 1
        elif y[i] == 0 and y_pred[i] == 1:
            FP += 1

acc = (TP + TN) / (TP + FP + TN + FN)
rec = TP / (TP + FN)
prec = TP / (TP + FP)
acc, rec, prec

(0.7209636821287306, 0.6876708387282405, 0.7366312220681153)

In [50]:
torch.save(model, f'./models/net_acc{str(acc)[:4]}.pt')
